[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/wing/wing.ipynb)

# Lift without vorticity

In an incompressible, inviscid flow past a wing the velocity has no divergence, since the fluid cannot be compressed, and no vorticity, since without viscosity none is ever made. The divergence and the vorticity are the scalar and the bivector part of the velocity's geometric derivative, so the flow obeys one equation: that derivative is zero. To compute it, we take the velocity's gradient, the map from a small step to the change in velocity, and contract it against an open vector. The derivative vanishes everywhere around the wing, and still the wing is lifted: the circulation that lifts it lives in the loop around the wing, not at any point of the flow.

In the notation of complex analysis this is the classical route of Joukowski: the complex potential of the flow past a cylinder, analytic by the equations of Cauchy and Riemann, carried to the wing by his map $z = \zeta + c^2/\zeta$, with the velocity the conjugate of its derivative and the lift given by the Kutta–Joukowski theorem.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from examples.mechanics.wing import core, render
from examples.mechanics.wing.core import Flow, Wing, ga, pitched, rings

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
VelocityGradient = ga.gatype((Vector, Vector))                               # Vector <- Vector

# The wing level along the stream; the cylinder's centre before the map sets its thickness and camber.
thickness, camber, scale = 0.1, 0.08, 1.0
speed, density = 1.0, 1.0
attack = np.deg2rad(8.0)
rings_count, angles, reach = 140, 360, 6.0
# The grid of circles drawn: centres of the circles through both critical points, across the chord
# in steps of the critical point's distance, and distance ratios of the circles around one of them;
# points along each circle; and the frames of the cylinder's deformation into the wing.
through_offsets = np.array([-1.5, -0.6, 0.0, 0.6, 1.5])
distance_ratios = np.array([0.2, 0.45, 0.7])
outline_points = 400
morph_frames, morph_duration_ms = 24, 80

## 1. The cylinder and its circulation

The wing starts as a cylinder. A map, built in section 3, will fold the cylinder's surface at one point into the wing's sharp trailing edge, and the flow can only leave that edge smoothly if it stands still there before the map. One circulation around the cylinder makes it so, the Kutta condition, and it comes out as a bivector: the stream wedged with the direction from the cylinder's centre to the trailing edge, `4 * pi * (stream ^ edge)`. The lift is the circulation contracted with the stream, times the density.

In scalar notation the lift reads as $L = \rho U \Gamma$, with $\Gamma = 4\pi U R \sin(\alpha + \beta)$ for a cylinder of radius $R$ whose trailing edge lies at the angle $\beta$ below its centre.

In [ ]:
# The wing level along the stream, then pitched nose up by the angle of attack: the rotor turns the
# cylinder's centre and the chord together, and the map built on the chord turns with them.
wing = pitched(Wing(-thickness * mv.x + camber * mv.y, scale, mv.x), attack)   # [] Wing
stream = speed * mv.x                                                          # [] Vector
# From the cylinder's centre to the trailing edge, where the flow must stand still.
edge = wing.scale * wing.chord - wing.centre                                   # [] Vector
# The circulation that stops it there, and the lift it gives.
swirl = 4 * np.pi * (stream ^ edge)                                            # [] Bivector
lift = density * (swirl | stream)                                              # [] Vector

## 2. The velocity past the cylinder

Past the cylinder the velocity is three terms: the stream itself; less its image in the cylinder, the stream sandwiched between two inverses of the position and scaled by the radius squared, which cancels the flow through the cylinder's surface; and the swirl, the circulation times the position's inverse. A small step changes the position's inverse by minus the step sandwiched between two inverses, and that gives the velocity's gradient map, `Vector <- Vector`. Contracted against an open vector it is the velocity's derivative, and it vanishes everywhere: no divergence, no vorticity.

A step also changes two fields that describe the flow: the velocity potential by `velocity | step`, how far the step goes along the flow, and the stream function, a bivector, by `velocity ^ step`, how much flow crosses it. The two are the parts of one product, `velocity * step`. The stream function is constant along each streamline, and it is written out below.

In [ ]:
def cylinder(centre: Vector, edge: Vector, swirl: Bivector, stream: Vector, plane: Vector) -> tuple[Vector, VelocityGradient, Bivector]:
    """The flow past a cylinder through `centre + edge`: its velocity, the velocity's gradient map,
    and the stream function."""
    offsets = plane - centre                                                   # [...] Vector
    radius_squared = edge | edge                                               # [] Scalar
    inverse = offsets.inverse()                                                # [...] Vector
    # The stream, less its image in the cylinder, and the swirl around it.
    velocity = stream - radius_squared * (inverse * stream * inverse) - swirl * inverse / (2 * np.pi)   # [...] Vector
    # A step changes the inverse by minus the step between two inverses, and the velocity with it.
    change = -(inverse * Vector * inverse)                                     # [...] Vector <- Vector
    gradient = -radius_squared * (change * stream * inverse + inverse * stream * change) - swirl * change / (2 * np.pi)
    # The stream function: the flow crossing from the centre's line along the stream, the image's,
    # and the swirl's, which grows with the logarithm of the distance.
    flux = (stream ^ offsets) + radius_squared * (inverse ^ stream) - swirl * (offsets | offsets).log() / (4 * np.pi)
    return velocity, gradient, flux


# Points on rings around the cylinder, from its surface out.
plane = rings(wing, rings_count, angles, reach)                                # [rings, angles] Vector
velocity, gradient, flux = cylinder(wing.centre, edge, swirl, stream, plane)
# The velocity's derivative: the open vector times the gradient's change, contracted.
derivatives = (Vector * gradient(Vector)).contract(1, 2)                       # [rings, angles] Even
print("largest divergence or vorticity past the cylinder:", np.abs(derivatives.kernel).max())

## 3. The map

The Joukowski map turns the cylinder into the wing. It adds to each point its inverse reflected in the line of a critical point, `p + (critical >> p.inverse())`. A small step moves under the map by its Jacobian: the step, less the step turned and scaled by the even multivector `critical * p.inverse()`. Turning and scaling keeps angles, so the map keeps the angle at which any two curves cross. The exceptions are `critical` and `-critical`, where `critical * p.inverse()` is one or minus one and the Jacobian vanishes; there the map doubles angles instead, and moves the two points to `2 * critical` and its negative.

The figure shows what the map does to two families of circles. Circles through both critical points land on circles through their images; circles around one critical point, the points at one ratio of distances to the two, land on circles around its image, at the ratio squared. The circle centred between the critical points crosses through both at right angles, which double to straight ones: it flattens onto the plate between the images. The wing's circle passes through `critical` and goes around `-critical`. Where it passes through, the doubled angle folds its smooth edge into a cusp, the sharp trailing edge.

In the notation of complex analysis the map reads as $z = \zeta + c^2/\zeta$, the circles as the bipolar coordinates of the two critical points, and the doubling as $\frac{z - 2c}{z + 2c} = \left(\frac{\zeta - c}{\zeta + c}\right)^2$.

In [ ]:
def joukowski(critical: Vector, plane: Vector) -> Vector:
    """Each point plus its inverse reflected in the line of the critical point."""
    return plane + (critical >> plane.inverse())                               # [...] Vector


# The trailing edge is a critical point: the wing's circle passes through it.
critical = wing.scale * wing.chord                                             # [] Vector
turns = (mv.xy * np.linspace(0, np.pi, outline_points)).exp()                 # [outline_points] Rotor
# Circles through both critical points, centred on the line across the chord through the map's centre.
across = through_offsets * critical.dual()                                     # [through] Vector
reach_through = ((critical | critical) + (across | across)).square_root()      # [through] Scalar
through = across[:, None] + reach_through[:, None] * (turns >> wing.chord)[None, :]   # [through, outline_points] Vector
# Circles around each critical point, the points at one ratio of distances to the two: centred
# along the chord, beyond the critical point they surround.
ratios = np.concatenate([distance_ratios, 1 / distance_ratios])
centres = critical * ((1 + ratios**2) / (1 - ratios**2))                       # [around] Vector
reach_around = (critical | critical).square_root() * (2 * ratios / np.abs(1 - ratios**2))   # [around] Scalar
around = centres[:, None] + reach_around[:, None] * (turns >> wing.chord)[None, :]   # [around, outline_points] Vector
grid = concatenate([through, around])                                          # [circles, outline_points] Vector
wing_circle = wing.centre + (turns >> edge)                                    # [outline_points] Vector
render.bipolar(grid, joukowski(critical, grid), wing_circle, joukowski(critical, wing_circle), critical);

## 4. Carried to the wing

The flow at the wing is the cylinder's carried along: the velocity potential and the stream function take, at each point's image, the values they had at the point. That this is again an ideal flow is what keeping angles buys. The Jacobian turns and scales every step, so the carried velocity is the cylinder's, turned as the steps are and divided by the scale, and its divergence and vorticity are the cylinder's divided by the scale squared: still zero. Lines of one value of the stream function land on lines of one value, so the cylinder's surface, a streamline, lands on the wing's, and no flow crosses it; far away the map barely moves anything, and the flow is still the stream.

A step at the image is first undone by the Jacobian, so the potential's gradient map at the cylinder, `velocity | Vector`, composed with the inverse of the Jacobian, `jacobian.inverse()`, is its gradient map at the wing. The velocity at the wing is its derivative, the same contraction against an open vector, and the stream function's values at the images draw the wing's streamlines. In the picture the pressure is low where the flow is fast, over the wing, and the arrow is the lift.

In [ ]:
def flow(wing: Wing, swirl: Bivector, stream: Vector, plane: Vector, critical: Vector) -> Flow:
    velocity, _, flux = cylinder(wing.centre, wing.scale * wing.chord - wing.centre, swirl, stream, plane)
    # How the map moves a small step: less the step turned and scaled by the even `critical * p.inverse()`.
    jacobian = Vector - ((critical * plane.inverse()) >> Vector)              # [...] Vector <- Vector
    # The potential keeps its value at a point's image: a step there is first undone by the Jacobian.
    potential_gradient = (velocity | Vector)(jacobian.inverse())                # [...] Scalar <- Vector
    # The velocity at the wing: the potential's derivative.
    at_wing = (Vector * potential_gradient(Vector)).contract(1, 2)            # [...] Vector
    return Flow(joukowski(critical, plane), at_wing, potential_gradient, flux)


flows = flow(wing, swirl, stream, plane, critical)                             # [rings, angles] Flow
render.figure(flows, lift, speed);

Moving the critical points apart deforms the cylinder into the wing. While they lie inside it, the map is smooth on its surface and the body is round; as `critical` reaches the surface, its doubled angle there turns into the cusp. The yellow points are their images, `2 * critical` and its negative. They start together at the centre of the map, while the cylinder's centre, the white point, sits behind it by the thickness and above it by the camber: that offset is all that makes the wing differ front to back and top to bottom. The circulation is held at the wing's throughout, so the last frame is the wing's flow.

In [ ]:
# The critical points moved apart from the map's centre out to the trailing edge, and the flow at
# each step. The cylinder and its circulation stay put; only the map changes, the identity at first.
fractions = np.linspace(0.0, 1.0, morph_frames)
# Marked: where the map sends the critical points, twice as far out, and the cylinder's centre.
frames = [render.marked_frame(flow(wing, swirl, stream, plane, fraction * critical), lift, speed,
                              2 * fraction * stack([critical, -critical]), wing.centre)
          for fraction in fractions]
render.inline(frames, morph_duration_ms)

## 5. Around the wing

Around a ring about the wing, the velocity times each step, summed, adds up the changes of both fields once around: the potential's, the scalar part, and the stream function's, the bivector part. The stream function comes back to where it started, so the bivector part is zero, no net flow out of the ring. The potential does not: it gains the circulation. Every ring gives the same sum, because between any two rings the velocity's derivative vanishes; the circulation is held by the loop around the wing, not by any point of the flow.

The pressure on the surface pushes the wing by half the density times the squared speed, along the inward normal at each step. Summed around the wing, that force is the lift from the circulation, across the stream, and nothing along it: an ideal flow that lifts but does not drag, d'Alembert's paradox.

In the notation of vector calculus the sum's two parts read as the circulation $\oint \mathbf v \cdot d\mathbf l$ and the flux $\oint \mathbf v \times d\mathbf l$, and their agreement over every ring as the theorems of Stokes and Gauss.

In [ ]:
# Around every ring, the velocity times each step, the two ends of the step averaged.
next_idx = (np.arange(angles) + 1) % angles
steps = flows.points[:, next_idx] - flows.points                               # [rings, angles] Vector
loops = ((flows.velocity + flows.velocity[:, next_idx]) / 2 * steps).sum(axis=-1)   # [rings] Even
# The surface: the innermost ring. The pressure is low where the flow is fast: it pushes by half the
# density times the squared speed, averaged over each step, along the inward normal, the step turned
# a quarter turn back.
surface = flow(wing, swirl, stream, rings(wing, 1, angles, 1.0)[0], critical)  # [angles] Flow
surface_steps = surface.points[next_idx] - surface.points                      # [angles] Vector
speed_squared = surface.velocity | surface.velocity                            # [angles] Scalar
force = (0.5 * density * (speed_squared + speed_squared[next_idx]) / 2 * -surface_steps.dual()).sum(axis=-1)   # [] Vector

In [ ]:
print(f"around the nearest ring:\n{loops[0].coefficient_table()}")
print(f"around the farthest ring:\n{loops[-1].coefficient_table()}")
print(f"the circulation:\n{swirl.coefficient_table()}")
print(f"pressure force:\n{force.coefficient_table()}")
print(f"lift from the circulation:\n{lift.coefficient_table()}")

In [ ]:
# checks
# The notebook's flow is the core's.
reference = core.flow(wing, stream, plane, critical)
np.testing.assert_allclose((reference.velocity - flows.velocity).kernel, 0.0, atol=1e-12)
# The map sends the critical points to twice themselves.
np.testing.assert_allclose((joukowski(critical, critical) - 2 * critical).kernel, 0.0, atol=1e-12)
# A circle around a critical point lands on a circle around its image, at its distance ratio squared.
landed = joukowski(critical, around)                                           # [around, outline_points] Vector
distance_ratio = ((landed - 2 * critical).norm() / (landed + 2 * critical).norm()).to_array()
np.testing.assert_allclose(distance_ratio, (ratios**2)[:, None] * np.ones(outline_points), rtol=1e-9)
# The velocity has no derivative anywhere past the cylinder.
np.testing.assert_allclose(derivatives.kernel, 0.0, atol=1e-12)
# A step changes the stream function by the velocity's flux through it, at the wing as at the cylinder.
step = mv.vector([0.3, -0.2]) * 1e-6                                           # [] Vector
forward, backward = flow(wing, swirl, stream, plane + step, critical), flow(wing, swirl, stream, plane - step, critical)
crossing = (forward.points - backward.points) / 2                              # [rings, angles] Vector
np.testing.assert_allclose(((forward.stream - backward.stream) / 2 - (flows.velocity ^ crossing)).kernel, 0.0, atol=1e-12)
# Every ring gives the same sum, the circulation with no flux, and the pressure force is the lift,
# to the resolution of the rings.
np.testing.assert_allclose(loops.kernel - loops.kernel[:1], 0.0, atol=1e-11)
np.testing.assert_allclose((loops - swirl.dual()).kernel, 0.0, atol=1e-3)
np.testing.assert_allclose((force - lift).kernel, 0.0, atol=1e-3)